# Colab MCP Server — Remote Dev for Claude

This notebook turns your Colab session into a live **MCP (Model Context Protocol) server**
that Claude Code can connect to. Once running, Claude can:

- Read, write, and search files anywhere in `/content`
- Execute Python code directly in **this kernel** (variables persist between calls)
- Run shell commands (`apt install`, `wget`, `git`, etc.)
- Install packages on-the-fly
- Navigate and inspect the filesystem

## Quick start
1. **Cell 1** — install deps (run once)
2. **Cell 2** — define the MCP server
3. **Cell 3** — launch + ngrok (paste your token, then run)
4. Copy the printed `claude mcp add` command into your local terminal

> **Runtime tip:** No GPU needed for the server itself.
> Switch to GPU only if your project code requires it.

---
## Cell 1 — Install dependencies

In [ ]:
!pip install -q fastmcp pyngrok
print("Dependencies ready.")

---
## Cell 2 — Define the MCP server

All tools are registered here. **Re-run this cell** any time you want to add or
modify a tool without restarting the server.

In [ ]:
import os, io, sys, shutil, traceback, subprocess, contextlib
from pathlib import Path
from typing import Optional
from concurrent.futures import ThreadPoolExecutor, TimeoutError as _FuturesTimeout

from fastmcp import FastMCP

# ---------------------------------------------------------------------------
# Shared execution namespace.
# Attaching to get_ipython().user_ns means:
#   - Variables defined in notebook cells are visible to Claude's execute_python
#   - Variables Claude defines appear in this notebook's namespace too
# ---------------------------------------------------------------------------
try:
    _ns = get_ipython().user_ns  # noqa: F821  — IPython global
except NameError:
    _ns = {"__builtins__": __builtins__}

mcp = FastMCP("Colab Remote Dev")


# ===========================================================================
#  FILESYSTEM TOOLS
# ===========================================================================

@mcp.tool()
def read_file(path: str, encoding: str = "utf-8") -> str:
    """Return the full text content of a file."""
    return Path(path).expanduser().read_text(encoding=encoding)


@mcp.tool()
def write_file(path: str, content: str, encoding: str = "utf-8") -> str:
    """Write (overwrite) a file with the given content. Creates parent dirs if needed."""
    p = Path(path).expanduser()
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding=encoding)
    return f"Written {p.stat().st_size} bytes -> {p.resolve()}"


@mcp.tool()
def append_file(path: str, content: str, encoding: str = "utf-8") -> str:
    """Append content to a file (creates it if it does not exist)."""
    p = Path(path).expanduser()
    p.parent.mkdir(parents=True, exist_ok=True)
    with p.open("a", encoding=encoding) as f:
        f.write(content)
    return f"Appended {len(content)} chars -> {p.resolve()}"


@mcp.tool()
def list_directory(path: str = ".") -> list:
    """List all entries in a directory. Returns name, path, type (file/dir), and size."""
    p = Path(path).expanduser()
    entries = []
    for item in sorted(p.iterdir()):
        try:
            stat = item.stat()
            entries.append({
                "name": item.name,
                "path": str(item.resolve()),
                "type": "dir" if item.is_dir() else "file",
                "size_bytes": stat.st_size,
            })
        except PermissionError:
            entries.append({"name": item.name, "type": "unknown", "error": "permission denied"})
    return entries


@mcp.tool()
def create_directory(path: str) -> str:
    """Create a directory (and all parents). Safe to call if it already exists."""
    Path(path).expanduser().mkdir(parents=True, exist_ok=True)
    return f"Created: {Path(path).expanduser().resolve()}"


@mcp.tool()
def delete_path(path: str) -> str:
    """Permanently delete a file or a directory tree."""
    p = Path(path).expanduser()
    if p.is_dir():
        shutil.rmtree(p)
        return f"Directory deleted: {p}"
    p.unlink()
    return f"File deleted: {p}"


@mcp.tool()
def copy_path(src: str, dst: str) -> str:
    """Copy a file or directory tree from src to dst."""
    s, d = Path(src).expanduser(), Path(dst).expanduser()
    if s.is_dir():
        shutil.copytree(s, d)
    else:
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(s, d)
    return f"Copied: {s} -> {d}"


@mcp.tool()
def move_path(src: str, dst: str) -> str:
    """Move or rename a file or directory."""
    shutil.move(src, dst)
    return f"Moved: {src} -> {dst}"


@mcp.tool()
def file_info(path: str) -> dict:
    """Return metadata (exists, type, size, mtime) for a path."""
    p = Path(path).expanduser()
    if not p.exists():
        return {"exists": False, "path": str(p)}
    stat = p.stat()
    return {
        "exists": True,
        "path": str(p.resolve()),
        "type": "dir" if p.is_dir() else "file",
        "size_bytes": stat.st_size,
        "modified_epoch": stat.st_mtime,
        "is_symlink": p.is_symlink(),
    }


@mcp.tool()
def find_files(pattern: str, root: str = ".") -> list:
    """
    Recursively find files matching a glob pattern under root.
    Examples: pattern='**/*.py', pattern='*.csv', pattern='**/config.*'
    """
    return sorted(str(p) for p in Path(root).expanduser().rglob(pattern))


@mcp.tool()
def search_in_files(
    text: str,
    root: str = ".",
    file_pattern: str = "*",
    case_sensitive: bool = True,
    max_results: int = 200,
) -> list:
    """
    Search for a text string in files under root matching file_pattern.
    Returns [{file, line_number, line}, ...] for every matching line.
    """
    matches = []
    needle = text if case_sensitive else text.lower()
    for fpath in Path(root).expanduser().rglob(file_pattern):
        if not fpath.is_file() or len(matches) >= max_results:
            break
        try:
            for i, line in enumerate(fpath.read_text(errors="replace").splitlines(), 1):
                haystack = line if case_sensitive else line.lower()
                if needle in haystack:
                    matches.append({
                        "file": str(fpath),
                        "line_number": i,
                        "line": line.rstrip(),
                    })
        except Exception:
            pass
    return matches


@mcp.tool()
def get_working_directory() -> str:
    """Return the current working directory of the Colab session."""
    return os.getcwd()


@mcp.tool()
def set_working_directory(path: str) -> str:
    """Change the current working directory of the Colab session."""
    os.chdir(path)
    return f"CWD -> {os.getcwd()}"


# ===========================================================================
#  CODE EXECUTION TOOLS
# ===========================================================================

def _run_code(code: str) -> dict:
    """Execute code in the shared namespace and capture all output."""
    stdout_buf = io.StringIO()
    stderr_buf = io.StringIO()
    result_val = None
    error = None

    with contextlib.redirect_stdout(stdout_buf), contextlib.redirect_stderr(stderr_buf):
        try:
            # Try eval first so bare expressions (e.g. '1 + 1', 'df.head()')
            # return their repr as 'result' rather than printing nothing.
            try:
                compiled = compile(code, "<mcp>", "eval")
                result_val = repr(eval(compiled, _ns))
            except SyntaxError:
                exec(compile(code, "<mcp>", "exec"), _ns)
        except Exception:
            error = traceback.format_exc()

    return {
        "stdout": stdout_buf.getvalue(),
        "stderr": stderr_buf.getvalue(),
        "result": result_val,
        "error": error,
    }


@mcp.tool()
def execute_python(code: str, timeout: int = 60) -> dict:
    """
    Execute Python code in the live Colab kernel namespace.

    - Variables defined in this call persist and are visible in future calls
      and in the notebook cells themselves.
    - Returns: stdout (string), stderr (string), result (repr of last
      expression if code is a single expression), error (traceback string
      or null).
    - timeout: seconds before the call is aborted (default 60).
    """
    with ThreadPoolExecutor(max_workers=1) as pool:
        future = pool.submit(_run_code, code)
        try:
            return future.result(timeout=timeout)
        except _FuturesTimeout:
            return {
                "stdout": "",
                "stderr": "",
                "result": None,
                "error": f"TimeoutError: execution exceeded {timeout}s — use a larger timeout or restructure the code.",
            }


@mcp.tool()
def install_package(packages: str) -> dict:
    """
    Install one or more PyPI packages with pip.
    packages: space-separated names, e.g. 'numpy pandas matplotlib'.
    Returns stdout/stderr and whether the install succeeded.
    """
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + packages.split()
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    return {
        "returncode": r.returncode,
        "stdout": r.stdout[-3000:],
        "stderr": r.stderr[-3000:],
        "success": r.returncode == 0,
    }


@mcp.tool()
def run_shell(command: str, cwd: Optional[str] = None, timeout: int = 60) -> dict:
    """
    Run a shell (bash) command and return stdout, stderr, and exit code.
    Full shell features work: pipes, redirects, &&, environment variables.
    cwd: working directory for the command (defaults to current).
    """
    r = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True,
        cwd=cwd,
        timeout=timeout,
    )
    return {
        "returncode": r.returncode,
        "stdout": r.stdout,
        "stderr": r.stderr,
        "success": r.returncode == 0,
    }


@mcp.tool()
def list_variables() -> dict:
    """
    List all user-defined variables currently in the execution namespace.
    Returns {name: type_name} for every non-dunder, non-private variable.
    """
    skip = {
        "In", "Out", "get_ipython", "exit", "quit", "open",
        "__builtin__", "__builtins__", "_ns", "mcp",
    }
    return {
        k: type(v).__name__
        for k, v in _ns.items()
        if not k.startswith("_") and k not in skip
    }


@mcp.tool()
def reset_namespace() -> str:
    """Clear all user-defined variables from the shared execution namespace."""
    protected = {k for k in _ns if k.startswith("__")}
    for k in list(_ns.keys()):
        if k not in protected:
            del _ns[k]
    return "Execution namespace cleared."


# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
try:
    tools = mcp.list_tools()
    print(f"MCP server defined — {len(tools)} tools registered:")
    for t in tools:
        desc = (t.description or "").splitlines()[0][:55]
        print(f"  {t.name:<30} {desc}")
except Exception:
    print("MCP server defined. (Tool listing not available in this fastmcp version.)")

---
## Cell 3 — Launch server and expose via ngrok

Get your **free ngrok auth token** at: https://dashboard.ngrok.com/get-started/your-authtoken  
Paste it in `NGROK_TOKEN` below, then run this cell.

In [ ]:
import threading, time
from pyngrok import ngrok

# ── Configuration ─────────────────────────────────────────────────────────────
NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"   # <-- replace with your token
MCP_PORT    = 8080

# ── Start MCP server in a background daemon thread ────────────────────────────
# fastmcp's run() blocks, so we run it off the main thread.
# The daemon flag means it dies automatically when the notebook session ends.
_server_kwargs = dict(transport="sse", host="0.0.0.0", port=MCP_PORT)
_server_thread = threading.Thread(
    target=mcp.run, kwargs=_server_kwargs, daemon=True, name="mcp-server"
)
_server_thread.start()
time.sleep(3)   # let uvicorn finish binding

if not _server_thread.is_alive():
    raise RuntimeError(
        "MCP server thread exited immediately — see any tracebacks above. "
        "Most likely cause: port 8080 already in use (re-run Cell 3)."
    )

# ── Open ngrok tunnel ─────────────────────────────────────────────────────────
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(MCP_PORT, "http")
MCP_PUBLIC_URL = tunnel.public_url
SSE_URL = f"{MCP_PUBLIC_URL}/sse"

# ── Print connection instructions ─────────────────────────────────────────────
sep = "-" * 62
print(sep)
print("  Colab MCP Server is LIVE")
print(sep)
print(f"  SSE endpoint : {SSE_URL}")
print(sep)
print()
print("  Option A — one-liner (run in your local terminal):")
print(f"    claude mcp add colab --transport sse {SSE_URL}")
print()
print("  Option B — edit ~/.claude/claude.json manually:")
print("  {")
print('    "mcpServers": {')
print('      "colab": {')
print('        "type": "sse",')
print(f'        "url":  "{SSE_URL}"')
print('      }')
print('    }')
print('  }')
print()
print("  After adding, restart Claude Code (or run /mcp in a session).")
print(sep)

---
## Cell 4 — Smoke test (optional)

Call the tools directly from Python to confirm they work before connecting Claude.

In [ ]:
import json

# 1. Filesystem: write + read back
write_file("/content/mcp_test.txt", "hello from MCP\nline 2\n")
content = read_file("/content/mcp_test.txt")
print("read_file:", repr(content))

# 2. List /content
entries = list_directory("/content")
print(f"\nlist_directory: {len(entries)} entries")
for e in entries[:6]:
    print(f"  {e['type']:4}  {e['name']}")

# 3. Execute Python — expression
r = execute_python("2 ** 10")
print(f"\nexecute_python('2**10'): result={r['result']}  error={r['error']}")

# 4. Execute Python — statement with print
r = execute_python("x_mcp = [i**2 for i in range(5)]; print(x_mcp)")
print(f"execute_python (list comp): stdout={r['stdout'].strip()}")

# 5. Variable persists in namespace
r = execute_python("x_mcp")
print(f"execute_python ('x_mcp'):  result={r['result']}")

# 6. Shell command
r = run_shell("df -h / | tail -1")
print(f"\nrun_shell ('df -h'): {r['stdout'].strip()}")

# 7. List variables (should include x_mcp)
vars_ = list_variables()
print(f"\nlist_variables: x_mcp -> {vars_.get('x_mcp', 'NOT FOUND')}")

print("\nAll smoke tests passed.")

---
## Reference — Tool catalogue

| Tool | Category | What it does |
|---|---|---|
| `read_file(path)` | Filesystem | Read full text of a file |
| `write_file(path, content)` | Filesystem | Overwrite a file (creates dirs) |
| `append_file(path, content)` | Filesystem | Append to a file |
| `list_directory(path)` | Filesystem | List entries with sizes |
| `create_directory(path)` | Filesystem | Create dir tree |
| `delete_path(path)` | Filesystem | Delete file or dir tree |
| `copy_path(src, dst)` | Filesystem | Copy file or dir |
| `move_path(src, dst)` | Filesystem | Move / rename |
| `file_info(path)` | Filesystem | Size, type, mtime |
| `find_files(pattern, root)` | Filesystem | Recursive glob (`**/*.py`) |
| `search_in_files(text, root)` | Filesystem | Grep across files |
| `get_working_directory()` | Filesystem | Current CWD |
| `set_working_directory(path)` | Filesystem | Change CWD |
| `execute_python(code, timeout)` | Execution | Run Python in this kernel |
| `install_package(packages)` | Execution | `pip install` |
| `run_shell(command, cwd)` | Execution | Run bash command |
| `list_variables()` | Execution | Namespace snapshot |
| `reset_namespace()` | Execution | Clear user variables |

---
## Keeping the session alive

Colab disconnects after ~90 min of UI inactivity. While Claude is actively calling
tools the session stays alive. If you need to keep it alive manually, run:

In [ ]:
# Run this cell to keep the session alive while you step away.
# Stop it (square button) before letting Claude take over.
import time, datetime

print("Keep-alive running. Stop this cell to hand control back to Claude.")
while True:
    print(datetime.datetime.now().strftime("%H:%M:%S"), "— server alive")
    time.sleep(60)